## 0 · One Kick, One Question

> **Match day. 20 metres from goal. One free kick.**
>
> Forget calculus for a moment. You already know how to improve this shot:
>
> 1. **Choose** an angle.
> 2. **Watch** what the ball does.
> 3. **Name the miss:** wall, over the bar, or wrong part of the net.
> 4. **Change the cause** in the direction that should reduce that miss.
> 5. Repeat until the correction becomes tiny.
>
> That loop is gradient descent.

The ball's centre must clear the 1.8 m wall, pass fully inside the 2.44 m goal frame, and arrive near a 1.10 m target at the back of the net. The animation below keeps the match visible while the hidden quantities change.

**Do not study the numbers yet. Watch their story:** one choice at the foot becomes a flight, the flight becomes an impact, and the impact becomes feedback.

![Cinematic forward pass of the free kick with live motion and a restrained telemetry replay](images/free-kick-forward-telemetry.gif)

**What to notice:** the angle is chosen once; everything after it is a consequence. Calculus gives us a disciplined way to connect the final consequence back to that first choice.

<small>Match photograph: Michael Barera, [“Detroit City FC v. San Antonio FC 2023 20 (free kick)”](https://commons.wikimedia.org/wiki/File:Detroit_City_FC_v._San_Antonio_FC_2023_20_(free_kick).jpg), Wikimedia Commons, [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). Cropped, color-graded, and composited with physics-driven motion and instructional overlays.</small>

# Mathematical Foundations for Machine Learning

## The idea before the vocabulary

A model learns the way this striker improves:

> **Make a choice → observe the result → measure the miss → trace the miss back → adjust the choice.**

In this notebook, the first choice is one launch angle. We then expose a second knob — speed — so the scalar derivative becomes a gradient vector. In a neural network, that vector can contain millions of weight sensitivities.

| In the free kick | Plain-language job | Mathematical name |
| --- | --- | --- |
| Point toward something | Compare two directions | Dot product |
| Notice when rising becomes falling | Measure change at this instant | Derivative |
| Turn repeated misses into corrections | Improve a choice step by step | Gradient descent |
| Notice when a step becomes unstable | Measure how sharply loss bends | Curvature |
| Adjust angle and speed together | Coordinate many local instructions | Gradient vector |
| Combine many signals into new signals | Mix influences in parallel | Matrix multiplication |
| Decide what signals pass or saturate | Gate forward values and backward gradients | Activation function |
| Ask how the impact came from the controls | Pass and add sensitivities backward | Chain rule / backpropagation |
| Allow for imperfect execution | Optimize the probability of acceptable outcomes | Likelihood |

The notebook shows the physical idea first. Equations appear afterward in collapsible **“Name the pattern”** sections. Open them after the visual story makes sense.

In [ ]:
# Dependencies
import subprocess, sys

# Only install packages that are not already importable in this environment
required = [("numpy", "numpy"), ("scipy", "scipy")]
for imp, pkg in required:
    try:
        __import__(imp)
        print(f"  ok  {pkg}")
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
        print(f"  done {pkg}")

import numpy as np
from scipy import stats

np.random.seed(42)
print("All dependencies ready")

# Physical constants for the free kick scenario
g = 9.81          # gravity (m/s²)
v0 = 20.0        # launch speed (m/s)
BALL_RADIUS = 0.11
WALL_X = 9.15    # wall position (m)
WALL_H = 1.8     # wall height (m)
GOAL_X = 20.0    # front of the goal (m)
CROSS_H = 2.44   # crossbar height (m)
NET_X = 21.5     # back of the net (m)
TARGET_H = 1.10  # intended ball-centre height at back-net impact (m)
GOAL_BOTTOM = BALL_RADIUS
GOAL_TOP = CROSS_H - BALL_RADIUS


def ball_state(x, theta_deg):
    """Return time, height, and velocity when the ball reaches horizontal position x."""
    theta = np.radians(theta_deg)
    vx = v0 * np.cos(theta)
    t = x / vx
    vy = v0 * np.sin(theta) - g * t
    y = v0 * np.sin(theta) * t - 0.5 * g * t**2
    return {"x": float(x), "t": float(t), "y": float(y), "vx": float(vx), "vy": float(vy)}


def ball_height(x, theta_deg):
    """Height of the ball centre at horizontal position x and launch angle theta (degrees)."""
    return ball_state(x, theta_deg)["y"]


print("\nFree kick setup:")
print(f"  Launch speed: {v0} m/s")
print(f"  Wall at {WALL_X}m, ball centre must clear {WALL_H + BALL_RADIUS:.2f}m")
print(f"  Goal plane at {GOAL_X}m, legal centre window [{GOAL_BOTTOM:.2f}, {GOAL_TOP:.2f}]m")
print(f"  Back-net target at ({NET_X:.1f}m, {TARGET_H:.2f}m)")

---

## Part 1 — Which Way Are We Pointing?

Before the ball can travel well, the kick direction must agree with the intended direction.

Imagine two arrows laid on top of each other:

- same direction → they strongly agree,
- sideways to each other → they do not help each other,
- opposite directions → one fights the other.

A dot product compresses that directional agreement into one number. You already understand the geometry; the operation only gives it a score.

#### Predict first

The kick points at 25°. The intended direction is 20°. They differ by only 5°.

- **(a)** Almost complete agreement
- **(b)** Half agreement
- **(c)** They oppose each other

Make the visual prediction first, then run the code.

<details>
<summary><strong>Name the pattern: dot product</strong></summary>

For unit vectors, the score is the cosine of the angle between them:

$$\mathbf{a}\cdot\mathbf{b}=\cos(\text{angle between them})$$

The expanded coordinate formula is $a_1b_1+a_2b_2$. A dense layer repeats this weighted-alignment calculation across many inputs.

</details>

> **Intuition first:** Imagine rotating your kick direction toward the goal. At exactly the right angle they point the same way — that's maximum overlap. At 90° they're perpendicular — zero overlap. The dot product captures that overlap as a single number: 1 when perfectly aligned, 0 when perpendicular, -1 when opposite.

**Vector alignment visualised:** The dot product measures how much two vectors point in the same direction.

### Pause the picture in your head

Hold the kick arrow fixed and rotate the wind arrow around it.

| Wind direction | What you would feel | Alignment score |
| --- | --- | --- |
| Behind the kick | A push in the same direction | Large and positive |
| Sideways | Drift, but no forward help | Near zero |
| Into the kick | Resistance | Negative |

The score changes smoothly as the arrow rotates. Nothing magical happens at a particular angle; the number simply tracks how much one direction lies along the other.

#### Say it without notation

> The dot product answers: **“How much of this direction helps that direction?”**

Now run the next short cell. It calculates the score for a 25° kick and a 20° intended direction.

In [ ]:
#  Part 1: Dot products and vector alignment
# Build 2D unit vectors for the actual kick angle and the ideal goal-facing angle
kick_dir = np.array([np.cos(np.radians(25)), np.sin(np.radians(25))])  # 25° angle
goal_dir = np.array([np.cos(np.radians(20)), np.sin(np.radians(20))])  # ideal 20°

# Dot product of two unit vectors equals the cosine of the angle between them
dot = np.dot(kick_dir, goal_dir)
alignment = np.degrees(np.arccos(np.clip(dot, -1, 1)))

print(f"Kick direction (25°): {kick_dir.round(3)}")
print(f"Ideal direction (20°): {goal_dir.round(3)}")
print(f"Dot product: {dot:.4f}")
print(f"Angular difference: {alignment:.1f}°")
print()
print("Dot product in ML: W · x = weighted sum of features")
print("  'How much does each feature contribute to the prediction?'")
print("  A dot product is the core operation in every Dense layer.")

#### What just happened — and what's missing

The dot product between kick (25°) and goal (20°) is very close to 1 — confirming prediction (a). Five degrees of misalignment barely dents the alignment score because cosine is very flat near zero.

**This is the most-used operation in all of ML.** Every `layers.Dense` layer computes $\mathbf{W} \cdot \mathbf{x}$ — a dot product of the weight row with the input — for every output neuron, simultaneously. Attention scores in transformers are dot products of query and key vectors. If you understand "dot product = measure of alignment," you understand the core computation of every model in this curriculum.

**Missing piece:** The dot product tells us _how aligned_ the kick is, but not _when the ball peaks_ — and we need the peak to know whether it clears the wall. For that we need derivatives.

---

#### Your turn — dot product geometry

```python
# CHANGE: try kick_angle = 90 (straight up). What dot product with goal_dir (20°) do you predict?
# Then try kick_angle = 200. What happens when the kick goes backward?
kick_angle = 90   # degrees — change this
goal_angle = 20   # fixed

kick = np.array([np.cos(np.radians(kick_angle)), np.sin(np.radians(kick_angle))])
goal = np.array([np.cos(np.radians(goal_angle)), np.sin(np.radians(goal_angle))])
dp = np.dot(kick, goal)
print(f"kick at {kick_angle}° vs goal at {goal_angle}°  →  dot product = {dp:.4f}")
```

---

## Part 2 — The Instant the Ball Stops Rising

Watch a ball rise in slow motion.

- Early in the flight, each frame is noticeably higher than the last.
- Near the top, the height gained per frame shrinks.
- For one instant, the next frame is neither higher nor lower.
- After that, each frame drops.

That **height gained per step** is the derivative. Positive means rising. Negative means falling. Zero marks the turn.

#### Predict first

For the 25° shot, where does the turn happen?

1. Around the wall at 10 m
2. Between the wall and goal at 15–16 m
3. At the goal line near 20 m

The code samples the flight and then checks the exact turning point.

<details>
<summary><strong>Name the pattern: derivative</strong></summary>

The trajectory has a height function $h(x)$. Its derivative $dh/dx$ means “how many metres of height change for the next metre forward.” At the peak:

$$\frac{dh}{dx}=0$$

The full projectile formula lives in the code. The intuition to keep is simpler: **a derivative is a local change meter**.

</details>

In [ ]:
# Part 2: derivative to find the peak height
theta_deg = 25.0
theta_rad = np.radians(theta_deg)

# Sample the trajectory at 200 points between launch and the back of the net
x_vals = np.linspace(0, NET_X, 200)
heights = [ball_height(x, theta_deg) for x in x_vals]

# The peak is where dh/dx = 0; argmax on dense samples provides a numerical check
peak_idx = np.argmax(heights)
peak_x = x_vals[peak_idx]
peak_h = heights[peak_idx]

# Closed-form peak location from dh/dx = 0
x_peak_analytic = v0**2 * np.sin(2 * theta_rad) / (2 * g)
wall_state = ball_state(WALL_X, theta_deg)
goal_state = ball_state(GOAL_X, theta_deg)
net_state = ball_state(NET_X, theta_deg)

print(f"At launch angle {theta_deg}°:")
print(f"  Sampled peak: x={peak_x:.2f}m, y={peak_h:.2f}m")
print(f"  Analytic peak: x={x_peak_analytic:.2f}m")
print()
print(f"  Ball centre at wall: {wall_state['y']:.2f}m  "
      f"(need > {WALL_H + BALL_RADIUS:.2f}m: {'PASS' if wall_state['y'] > WALL_H + BALL_RADIUS else 'FAIL'})")
print(f"  Ball centre at goal: {goal_state['y']:.2f}m  "
      f"(legal [{GOAL_BOTTOM:.2f}, {GOAL_TOP:.2f}]m: {'PASS' if GOAL_BOTTOM < goal_state['y'] < GOAL_TOP else 'FAIL'})")
print(f"  Predicted back-net height: {net_state['y']:.2f}m  (target {TARGET_H:.2f}m)")
print()
print("Prediction check: answer 2 — the ball peaks between wall and goal.")
print("The derivative locates the turning point; the constraints tell us whether that curve scores.")

### Freeze the flight at three moments

| Moment | What the ball is doing | What that tells us |
| --- | --- | --- |
| Over the wall | Still rising | Clearing the wall does not guarantee a goal |
| At 15.6 m | Neither rising nor falling | This is the peak: local change is zero |
| At the goal | Falling, but still 3.35 m high | The shot turns downward too late to fit under the bar |

#### Build the causal picture

A higher peak is not automatically better. Moving the peak changes the entire second half of the flight. What matters is the ball's state **at each constraint**, not one impressive number in isolation.

> The derivative helps us locate the turn. The match constraints decide whether the resulting curve is useful.

#### Read the shot like a coach

The 25° ball turns at about **15.6 m**. That tells us the shape of the flight, but not whether it scores.

Now inspect the important moments:

- **At the wall:** 3.02 m — safely over.
- **At the goal plane:** 3.35 m — over the crossbar.
- **Verdict:** good first constraint, failed second constraint.

The useful lesson is not “the derivative solved the kick.” It is:

> **The derivative tells us how behavior is changing. The loss tells us whether that behavior serves the goal.**

To improve the shot, we now ask a practical question: if a slightly steeper angle makes the miss worse, which way should the next angle move?

<details>
<summary><strong>Compact rule</strong></summary>

If increasing the angle increases the loss, then $dL/d\theta>0$. The correction must move in the opposite direction, so the angle decreases.

</details>

#### Your turn

Try 35°, then 15°. Before running the code, say the miss out loud: **wall**, **high**, **low**, or **target**. Classification before calculation builds the right intuition.

---

## Part 3 — Learn by Missing

A 35° kick sails high. What information is hidden in that miss?

- The wall is already safe, so no correction is needed there.
- The ball crosses above the frame, so “higher” is the wrong direction.
- The back-net impact is also high, so the same angle change improves both errors.

We turn those observations into a **wrongness score** called loss. A worse miss gets a larger score. A legal shot near the target gets a score near zero.

Then we run one tiny experiment:

> If I nudge the angle upward, does wrongness rise or fall?

That answer is the gradient. If upward makes things worse, step downward. Repeat.

#### Predict first

Starting from 35°, what should happen?

1. The angle moves down toward ~19°
2. The angle moves up because the gradient is positive
3. The angle stays put because it already clears the wall

Watch each attempted kick below. The faded trails are memory: every miss becomes evidence for the next correction.

<details>
<summary><strong>Name the pattern: loss, gradient, update</strong></summary>

The loss combines four plain penalties: wall collision, below-frame miss, above-frame miss, and distance from the back-net target. Only violated constraints contribute.

The update is:

$$\text{new angle}=\text{old angle}-\text{learning rate}\times\text{gradient}$$

The minus sign means “move opposite the direction that makes wrongness grow.”

</details>

![Cinematic sequence of repeated free kicks where each visible miss produces the next angle correction](images/gradient-descent-free-kick.gif)

**Follow the verbs:** shoot → judge → correct. The numbers confirm the story; they are not the story.

In [ ]:
# Part 3: gradient descent on the physically complete free-kick loss
def kick_loss(theta_deg):
    """Wall + goal-frame constraints plus back-net target error; lower is better."""
    h_wall = ball_height(WALL_X, theta_deg)
    h_goal = ball_height(GOAL_X, theta_deg)
    h_net = ball_height(NET_X, theta_deg)

    wall_gap = max(0.0, WALL_H + BALL_RADIUS - h_wall)
    goal_low_gap = max(0.0, GOAL_BOTTOM - h_goal)
    goal_high_gap = max(0.0, h_goal - GOAL_TOP)
    net_error = h_net - TARGET_H

    return wall_gap**2 + 2 * goal_low_gap**2 + 2 * goal_high_gap**2 + 0.25 * net_error**2


learning_rate = 2.0
epsilon = 1e-4
theta = 35.0
history = [theta]

print(f"start: angle={theta:.3f}°  loss={kick_loss(theta):.6f}")

# Estimate the local slope, then move in the opposite direction
for step in range(36):
    gradient = (kick_loss(theta + epsilon) - kick_loss(theta - epsilon)) / (2 * epsilon)
    theta = np.clip(theta - learning_rate * gradient, 8.0, 45.0)
    history.append(float(theta))
    if step < 3 or (step + 1) % 6 == 0:
        print(f"step {step + 1:2d}: angle={theta:7.3f}°  dL/dθ={gradient:+9.5f}  loss={kick_loss(theta):.6f}")

final_angle = history[-1]
final_wall = ball_height(WALL_X, final_angle)
final_goal = ball_height(GOAL_X, final_angle)
final_net = ball_height(NET_X, final_angle)
is_scoreable = (
    final_wall > WALL_H + BALL_RADIUS
    and GOAL_BOTTOM < final_goal < GOAL_TOP
    and abs(final_net - TARGET_H) < 0.08
)

print(f"\nfinal angle: {final_angle:.3f}°")
print(f"wall={final_wall:.3f}m  goal={final_goal:.3f}m  net={final_net:.3f}m")
print(f"final loss={kick_loss(final_angle):.8f}  target hit={is_scoreable}")
print("Prediction check: answer 1 — a positive gradient caused a downward angle update.")

### Read the learning process without a graph

| Attempt | Angle | Wrongness | Coach's read |
| ---: | ---: | ---: | --- |
| 0 | 35.000° | 45.6755 | Far too high — make a large correction down |
| 1 | 21.133° | 0.1208 | Legal direction, still high — keep moving down |
| 6 | 20.189° | 0.0344 | Much closer — use a smaller correction |
| 18 | 19.348° | 0.0017 | Almost centered — correction is now tiny |
| 36 | 19.134° | 0.000018 | Target locked |

Two intuitions matter more than the curve:

1. **Large miss, large useful signal.** The first correction is dramatic because the high shot is unambiguous.
2. **Small miss, small useful signal.** Near the target, the gradient fades, so updates naturally become gentle.

The optimizer is not becoming tired or cautious. The local evidence is simply telling it that less correction remains.

---

## Why the Same Step Can Be Safe Here and Wild There

Think of steering:

- on a **gentle bend**, a large steering correction may still be safe;
- on a **hairpin bend**, the same correction can throw you across the road.

A loss function also bends. That local bend is called **curvature**.

At the 35° starting shot, wrongness curves upward fairly sharply. Our measured curvature is about **0.561**. For a locally bowl-shaped region, a useful stability warning is:

> Keep the learning rate below roughly **2 ÷ curvature**.

Here that boundary is about **3.56**:

- `0.2` is safe but timid,
- `2.0` is bold but still inside the local boundary,
- `5.0` is too aggressive and can bounce against the allowed angle range.

#### Predict first

Which learning rate should make the largest useful first correction without immediately crossing the local stability warning?

<details>
<summary><strong>Name the pattern: second derivative</strong></summary>

The gradient measures slope. Curvature measures how quickly that slope itself changes:

$$H(\theta)=\frac{d^2L}{d\theta^2}$$

For a simple quadratic bowl with curvature $H$, gradient descent is stable when $0<\alpha<2/H$. Real neural networks do not have one fixed curvature, so this is a local intuition, not a universal recipe.

</details>

In [ ]:
# Learning-rate intuition: estimate how sharply the loss bends
def loss_curvature(theta_deg, epsilon=1e-3):
    return (
        kick_loss(theta_deg + epsilon)
        - 2 * kick_loss(theta_deg)
        + kick_loss(theta_deg - epsilon)
    ) / epsilon**2


start_angle = 35.0
start_curvature = loss_curvature(start_angle)
local_stability_limit = 2 / start_curvature

print(f"At {start_angle:.1f}°:")
print(f"  local curvature H ≈ {start_curvature:.3f}")
print(f"  local stability warning 2/H ≈ {local_stability_limit:.2f}")
print()

for rate in [0.2, 2.0, 5.0]:
    test_angle = start_angle
    test_path = [test_angle]
    for _ in range(6):
        test_gradient = (kick_loss(test_angle + 1e-4) - kick_loss(test_angle - 1e-4)) / 2e-4
        test_angle = np.clip(test_angle - rate * test_gradient, 8.0, 45.0)
        test_path.append(float(test_angle))
    print(f"lr={rate:3.1f}: first={test_path[1]:6.2f}°  after 6={test_path[-1]:6.2f}°  loss={kick_loss(test_path[-1]):.4f}")

print("\nThe boundary changes as the angle moves because the loss bends differently elsewhere.")

#### What the optimizer actually knew

Gradient descent changed a visible high miss at 35° into a target-height shot near **19.13°**.

It did **not** know the final answer in advance. At each attempt it knew only:

1. how wrong the current shot was,
2. whether a slightly larger angle made that wrongness rise or fall,
3. how large a correction it was allowed to make.

That is enough.

The important bridge to machine learning is not “a ball rolls down a bowl.” It is:

> **A parameter caused an outcome. A loss judged the outcome. A gradient assigned direction to the cause.**

Our single angle can be tested by nudging it up and down. A neural network has millions of parameters, so repeating that experiment one parameter at a time would be impossibly slow. Part 5 shows how the chain rule reuses one forward pass to send responsibility backward efficiently.

---

## From One Knob to a Gradient Vector

A real kick has more than one adjustable cause. Let us expose two:

- **angle** changes the shape of the arc,
- **speed** changes how long gravity has to pull the ball down.

At `(30°, 16 m/s)`, ask two separate “what if?” questions:

1. If angle rises slightly while speed stays fixed, how does wrongness change?
2. If speed rises slightly while angle stays fixed, how does wrongness change?

The two answers form one gradient vector:

$$\nabla L=\begin{bmatrix}\partial L/\partial\text{angle}\\\partial L/\partial\text{speed}\end{bmatrix}$$

For the starting shot, both entries are negative. That means increasing either knob would reduce wrongness. Gradient descent updates **both knobs at the same time**.

#### Why two step scales?

A degree and a metre per second are different units. The same numeric step would not mean the same physical change. We therefore use one scale per knob:

- angle scale: `1.0`,
- speed scale: `0.1`.

This is the seed of an important future idea: optimizers such as Adam adapt the effective step scale for different parameters.

> A gradient is not one instruction repeated many times. It is a coordinated set of local instructions — one for every adjustable cause.

In [ ]:
# Two-parameter gradient descent: update angle and speed simultaneously
def ball_height_two_params(x, angle_deg, speed):
    angle_rad = np.radians(angle_deg)
    return x * np.tan(angle_rad) - g * x**2 / (2 * speed**2 * np.cos(angle_rad) ** 2)


def kick_loss_two_params(parameters):
    angle_deg, speed = parameters
    h_wall = ball_height_two_params(WALL_X, angle_deg, speed)
    h_goal = ball_height_two_params(GOAL_X, angle_deg, speed)
    h_net = ball_height_two_params(NET_X, angle_deg, speed)

    wall_gap = max(0.0, WALL_H + BALL_RADIUS - h_wall)
    goal_low_gap = max(0.0, GOAL_BOTTOM - h_goal)
    goal_high_gap = max(0.0, h_goal - GOAL_TOP)
    net_error = h_net - TARGET_H
    return wall_gap**2 + 2 * goal_low_gap**2 + 2 * goal_high_gap**2 + 0.25 * net_error**2


def two_parameter_gradient(parameters, epsilon=1e-4):
    gradient = np.zeros(2)
    for index in range(2):
        change = np.zeros(2)
        change[index] = epsilon
        gradient[index] = (
            kick_loss_two_params(parameters + change)
            - kick_loss_two_params(parameters - change)
        ) / (2 * epsilon)
    return gradient


parameters = np.array([30.0, 16.0])  # [angle in degrees, speed in m/s]
step_scales = np.array([1.0, 0.1])

print(f"start: angle={parameters[0]:.3f}°  speed={parameters[1]:.3f}m/s  loss={kick_loss_two_params(parameters):.6f}")
for step in range(20):
    gradient_vector = two_parameter_gradient(parameters)
    parameters = np.clip(
        parameters - step_scales * gradient_vector,
        [8.0, 12.0],
        [45.0, 26.0],
    )
    if step in [0, 1, 2, 5, 11, 19]:
        print(
            f"step {step + 1:2d}: angle={parameters[0]:7.3f}°  speed={parameters[1]:6.3f}m/s  "
            f"gradient=[{gradient_vector[0]:+.5f}, {gradient_vector[1]:+.5f}]  "
            f"loss={kick_loss_two_params(parameters):.6f}"
        )

print("\nBoth values changed on every update; neither waited for the other to finish.")

---

## Part 4 — From a Parameter Vector to Signal Mixing

We now have a parameter vector with two adjustable causes: angle and speed. A neural network can have millions.

The next question is different. During the forward pass, how do many incoming values combine into many new values?

Imagine a control desk:

- one row of dials creates predicted wall clearance,
- another creates predicted goal height,
- another could create scoring confidence.

Each dial says how strongly one input influences one output. A matrix stores the whole control desk, so all weighted combinations happen together.

#### Predict first

If every output uses some amount of angle and some amount of speed, what happens when angle changes?

- **(a)** Only one output can change
- **(b)** Every output connected to angle can change
- **(c)** Nothing changes because the bias absorbs it

<details>
<summary><strong>Name the pattern: matrix transformation</strong></summary>

$$\mathbf{y}=W\mathbf{x}+\mathbf{b}$$

- $\mathbf{x}$ is the input vector,
- each row of $W$ gathers the inputs for one output,
- $\mathbf{y}$ is the new representation.

Read it as: **mix all relevant inputs into every output that needs them**.

</details>

> **Picture it as a mixing desk:** each input feature has a dial leading into each output. The matrix stores all dial positions. Matrix multiplication applies the entire mix in one coordinated operation.

> **Bridge from the kick:** angle and speed are no longer isolated facts. They become inputs whose combination predicts several consequences. Learning means adjusting the mixing dials until those consequences become useful.

In [ ]:
#  Part 4: Matrix as a linear transformation
W = np.array([[2, 0.5], [-0.5, 1.5]])
b = np.array([0.1, -0.2])

features = np.array([0.4, 0.8])  # angle=40% of max, speed=80% of max

# This is exactly what a Dense/Linear layer computes: matrix-vector multiply plus bias
output = W @ features + b

print(f"Input features (angle, speed): {features}")
print(f"Weight matrix W:\n{W}")
print(f"Bias b: {b}")
print(f"Output W@x + b: {output.round(4)}")
print()
print("In a neural network:")
print("  features = pixel values / token embeddings / sensor readings")
print("  W = learned weights (what the model found useful)")
print("  output = the model's internal representation")
print()
print(f"Parameter count: W has {W.size} + b has {b.size} = {W.size + b.size} learnable values")
print("GPT-2's first attention layer: 768×2304 = 1,769,472 parameters (same operation, bigger numbers)")

### Walk one signal through the mixing desk

The runnable cell produced two outputs from two inputs: angle and speed.

Now imagine covering the **speed** column of the weight matrix with your hand. What remains is every route by which angle can affect the outputs. One angle value can push one output up strongly, push another down slightly, or be ignored entirely — depending on the learned dial for each route.

Then cover the **angle** column. You have isolated speed's influence.

This column-by-column view is often more intuitive than staring at the whole matrix:

- **a column follows one input** into every output,
- **a row gathers all inputs** for one output,
- the full multiplication performs every route together.

> A matrix is not an abstract grid. It is a map of who can influence whom, and by how much.

#### What just happened — and what's missing

The matrix **mixed every input feature into every output** — prediction (b). That's not a bug; it's the feature. The matrix learns which combinations of inputs matter for each output.

| Toy (this notebook)        | GPT-2 (production)                    |
| -------------------------- | ------------------------------------- |
| 2 input features           | 768 token embedding dimensions        |
| 2 output features          | 2304 (Q, K, V concatenated)           |
| 4 weight values            | 1,769,472 weight values               |
| `W @ features` in one line | `layers.Dense(2304)(x)` in one line   |

**The operation is identical. Only the shape changes.**

**Missing piece:** We can compute $y = Wx + b$ forward, but gradient descent needs $\frac{\partial L}{\partial W}$ — how does every single weight in $W$ affect the final loss? Computing that for a matrix with millions of entries requires the chain rule applied systematically.

---

#### Your turn — matrix rows control outputs independently

```python
# CHANGE W so that row 0 weights angle (col 0) heavily and ignores speed (col 1),
#    and row 1 does the opposite. Then double the angle feature and watch which output moves.
W_test = np.array([[3.0, 0.1],   # row 0: cares about angle, ignores speed
                   [0.1, 3.0]])  # row 1: cares about speed, ignores angle
b_test = np.array([0.0, 0.0])

features_normal = np.array([0.4, 0.8])
features_double = np.array([0.8, 0.8])  # double the angle feature only

out_normal = W_test @ features_normal + b_test
out_double = W_test @ features_double + b_test

print(f"Normal features {features_normal}: output = {out_normal.round(3)}")
print(f"Doubled angle  {features_double}: output = {out_double.round(3)}")
print(f"Delta: {(out_double - out_normal).round(3)}")
```

---

## Constraint Gates and Activation Functions

This notebook has already used two kinds of gates:

- `max(0, gap)` keeps a penalty asleep until a constraint is violated,
- `np.clip(angle, 8, 45)` prevents an update from leaving the allowed range.

Neural networks also need rules that decide what signals may pass forward and how gradients return backward. Those rules are **activation functions**.

![Animated comparison of hard clipping, ReLU gating, and sigmoid bounding as one signal sweeps from negative to positive](images/activation-functions-gates.gif)

### Read the three gates

| Gate | Forward intuition | Backward consequence |
| --- | --- | --- |
| **Hard clip** | Value hits a rigid lower or upper wall | Gradient is zero outside the legal interval; a parameter can stick at the boundary |
| **ReLU** | Negative evidence is blocked; positive evidence passes unchanged | Gradient is 0 on the negative side and 1 on the positive side |
| **Sigmoid** | Any real input is compressed smoothly into 0–1 | Gradient is largest near the middle and fades near 0 or 1 |

#### Why this matters later

A model is not only a stack of matrix multiplications. Without activations, many linear layers collapse into one larger linear transformation. Activations add gates and bends, allowing the network to represent conditional, non-linear behavior.

The animation also previews two future failure modes:

- a ReLU unit can become inactive when it remains on the blocked side,
- a sigmoid can learn slowly when it is pushed into a saturated extreme.

> Forward behavior and backward learnability are two views of the same gate.

---

## Part 5 — Start at the Net and Investigate Backward

The ball hits the back net **0.232 m too high**.

Before writing any derivative, reason like an investigator:

1. The impact is high, so the miss is positive.
2. Near this shot, making the angle steeper raises the impact.
3. Therefore a steeper angle would make the positive miss worse.
4. The next update should lower the angle.

That is the conclusion backpropagation must produce numerically.

The forward pass stored the chain of consequences:

**launch angle → flight → net height → target miss → wrongness**

The backward pass walks that chain in reverse. At each link it asks:

> “If this earlier value changed a little, how much would the final wrongness change?”

Watch the green signal travel from the impact back to the launch angle. The ball is not travelling backward in time; **responsibility is travelling backward through the recorded calculation**.

<details>
<summary><strong>Name the pattern: chain rule</strong></summary>

For the target branch:

$$\frac{dL}{d\theta}=\frac{dL}{d\text{miss}}\times\frac{d\text{miss}}{d\text{impact height}}\times\frac{d\text{impact height}}{d\theta}$$

At the animated shot, the local factors multiply to $+0.039809$ loss per degree. Positive means “a larger angle raises wrongness,” confirming the verbal prediction.

</details>

> **The intuition:** forward mode tells the story of consequences. Backward mode asks who was responsible. The gradient returned to an earlier value is its share of the final wrongness.

![Cinematic reverse-pass sequence that freezes the impact and carries responsibility back to the launch angle](images/backprop-free-kick.gif)

**Try to predict the final instruction before it appears:** because the ball is high and a steeper angle raises it, the update must lower the angle.

In [ ]:
# Part 5: explicit reverse-mode chain rule on the same free-kick loss
def dh_dtheta(x, theta_deg):
    """Derivative of ball height with respect to angle measured in degrees."""
    theta_rad = np.radians(theta_deg)
    sec2 = 1.0 / np.cos(theta_rad) ** 2
    a = g * x**2 / (2 * v0**2)
    dh_dradians = x * sec2 - 2 * a * sec2 * np.tan(theta_rad)
    return dh_dradians * np.pi / 180.0


def gradient_path_contributions(theta_deg):
    """Return each active path's contribution to dL/dtheta."""
    h_wall = ball_height(WALL_X, theta_deg)
    h_goal = ball_height(GOAL_X, theta_deg)
    h_net = ball_height(NET_X, theta_deg)

    wall_gap = max(0.0, WALL_H + BALL_RADIUS - h_wall)
    goal_low_gap = max(0.0, GOAL_BOTTOM - h_goal)
    goal_high_gap = max(0.0, h_goal - GOAL_TOP)
    net_error = h_net - TARGET_H

    return {
        "wall": -2 * wall_gap * dh_dtheta(WALL_X, theta_deg),
        "goal low": -4 * goal_low_gap * dh_dtheta(GOAL_X, theta_deg),
        "goal high": 4 * goal_high_gap * dh_dtheta(GOAL_X, theta_deg),
        "target": 0.5 * net_error * dh_dtheta(NET_X, theta_deg),
    }


def analytic_kick_gradient(theta_deg):
    """Backpropagate through every loss branch and add their contributions."""
    return sum(gradient_path_contributions(theta_deg).values())


# First trace: a legal shot where only the target path is active
trace_theta = final_angle + 0.65
trace_net_height = ball_height(NET_X, trace_theta)
trace_error = trace_net_height - TARGET_H
local_dloss_derror = 0.5 * trace_error
local_dheight_dtheta = dh_dtheta(NET_X, trace_theta)
chain_gradient = analytic_kick_gradient(trace_theta)
numerical_gradient = (kick_loss(trace_theta + 1e-4) - kick_loss(trace_theta - 1e-4)) / 2e-4

print("ONE ACTIVE PATH")
print(f"  forward angle:                 {trace_theta:.6f}°")
print(f"  forward back-net impact:       {trace_net_height:.6f}m")
print(f"  forward target error:          {trace_error:+.6f}m")
print(f"  backward dL/derror:            {local_dloss_derror:+.6f}")
print(f"  backward dh_net/dtheta:        {local_dheight_dtheta:+.6f} m/degree")
print(f"  chain-rule dL/dtheta:          {chain_gradient:+.6f}")
print(f"  finite-difference check:       {numerical_gradient:+.6f}")
print(f"  match within 1e-6:             {abs(chain_gradient - numerical_gradient) < 1e-6}")

# Second trace: a low shot where several consequences blame the same angle
branch_theta = 16.0
contributions = gradient_path_contributions(branch_theta)
branch_gradient = sum(contributions.values())
branch_numerical = (kick_loss(branch_theta + 1e-4) - kick_loss(branch_theta - 1e-4)) / 2e-4

print("\nMULTIPLE ACTIVE PATHS AT 16°")
for path_name, contribution in contributions.items():
    status = "active" if abs(contribution) > 1e-12 else "inactive"
    print(f"  {path_name:9s}: {contribution:+.6f}  ({status})")
print(f"  {'SUM':9s}: {branch_gradient:+.6f}")
print(f"  numerical: {branch_numerical:+.6f}")
print(f"  match within 1e-6: {abs(branch_gradient - branch_numerical) < 1e-6}")
print("\nBackpropagation adds every path that connects the loss to the same parameter.")

#### One cause can receive blame through several paths

For the animated legal shot, only the back-net target branch is active. Its reverse pass returns **+0.039809 loss per degree**:

> A larger angle would make this high impact worse. Lower the angle.

At 16°, the story branches:

- the ball can blame the angle through the **wall** constraint,
- through the **low edge of the goal frame**,
- through the **high edge** when relevant,
- and through the **back-net target**.

Each route returns its own signed contribution. Backpropagation **adds the contributions when the routes meet at the same parameter**.

| Path | Question asked backward | Contribution at 16° |
| --- | --- | ---: |
| Wall | How did angle affect wall clearance, then wall penalty? | Printed by the cell |
| Goal low | How did angle affect low-frame clearance, then its penalty? | Printed by the cell |
| Goal high | Is the high-frame penalty active? | Printed by the cell |
| Target | How did angle affect net height, then target error? | Printed by the cell |
| **Total** | What is angle's full effect on loss? | **Sum of every path** |

The finite-difference check verifies both the one-path and multi-path totals.

Frameworks such as TensorFlow and PyTorch automate this bookkeeping:

1. remember intermediate values while moving forward,
2. begin at the final wrongness,
3. pass local sensitivity backward through every route,
4. add contributions wherever routes merge.

<details>
<summary><strong>Why this scales</strong></summary>

Finite differences rerun the whole model after perturbing each parameter. Reverse-mode autodiff traverses the recorded graph once and reuses intermediate values. That is why millions of gradients can be computed in one backward pass.

</details>

---

## Part 6 — Why Optimising the Mean Is Not Enough

Our deterministic optimizer answered:

> “Which angle puts one perfect execution near the chosen back-net target?”

It found **19.13°**.

A noisy striker needs a different question:

> “Which intended angle puts the greatest share of imperfect executions inside the legal window?”

The legal interval is 18.44°–21.86°. With symmetric execution noise, its midpoint — **20.15°** — gives the largest long-run scoring probability.

Neither answer is wrong. They optimize different definitions of success:

- **target loss** values precision at one point,
- **likelihood loss** values probability mass across all acceptable outcomes.

#### Predict first

With 3° of typical variation, which aim scores more often?

- **(a)** 19.13°, because it is the deterministic optimum
- **(b)** 20.15°, because it centers the noisy attempts inside the legal interval
- **(c)** They must be identical

<details>
<summary><strong>Name the pattern: likelihood and negative log-likelihood</strong></summary>

We model executed angle $\Theta$ with a Gaussian around the intended aim. Scoring probability is:

$$P(18.44°<\Theta<21.86°)$$

To turn “make the observed outcome probable” into a quantity we minimize, use:

$$L_{NLL}=-\log P(\text{observed outcome})$$

Classification cross-entropy applies the same principle to discrete class probabilities produced by softmax.

</details>

> **Two different jobs:** optimisation chooses the intended action; probability describes how execution spreads around that intention. “Where should I aim?” and “How reliably can I do it?” are not the same question.

In [ ]:
# Part 6: compare a point-target objective with a likelihood objective
from scipy import stats

# Keep angles that clear the wall and cross fully inside the goal frame
scoreable_angles = [
    angle for angle in np.linspace(5, 60, 20_000)
    if (
        ball_height(WALL_X, angle) > WALL_H + BALL_RADIUS
        and GOAL_BOTTOM < ball_height(GOAL_X, angle) < GOAL_TOP
    )
]

if not scoreable_angles:
    raise RuntimeError("No launch angle satisfies the physical constraints")

theta_lo = min(scoreable_angles)
theta_hi = max(scoreable_angles)
probability_aim = (theta_lo + theta_hi) / 2
sigma = 3.0


def probability_of_scoring(intended_angle, execution_sigma=sigma):
    distribution = stats.norm(loc=intended_angle, scale=execution_sigma)
    return distribution.cdf(theta_hi) - distribution.cdf(theta_lo)


prob_target_aim = probability_of_scoring(final_angle)
probability_optimized = probability_of_scoring(probability_aim)
nll_target_aim = -np.log(prob_target_aim)
nll_probability_aim = -np.log(probability_optimized)
prob_score = probability_optimized

print(f"Legal angle window: [{theta_lo:.2f}°, {theta_hi:.2f}°]")
print(f"Execution variability: sigma={sigma:.1f}°\n")
print("TWO OBJECTIVES, TWO ANSWERS")
print(f"  target-height aim:     {final_angle:5.2f}°  P(score)={prob_target_aim:5.1%}  NLL={nll_target_aim:.3f}")
print(f"  probability-max aim:   {probability_aim:5.2f}°  P(score)={probability_optimized:5.1%}  NLL={nll_probability_aim:.3f}")
print(f"  likelihood gain:                         {probability_optimized - prob_target_aim:+5.1%}")
print("\nPrediction check: answer (b) — centering the noisy attempts raises probability mass inside the window.\n")

print("P(scoring) at the probability-maximizing aim:")
for sigma_test in [1.0, 2.0, 3.0, 5.0, 8.0, 10.0]:
    probability = probability_of_scoring(probability_aim, sigma_test)
    print(f"  sigma={sigma_test:4.1f}°: P(score)={probability:.1%}")

### Imagine 100 repeated kicks

At σ=3°, the two aims behave differently over many attempts:

| Strategy | Intended angle | Expected scores out of 100 | What it optimizes |
| --- | ---: | ---: | --- |
| Hit one back-net point precisely | 19.13° | About 41 | Distance from the target point |
| Center noisy attempts in the legal window | 20.15° | About 43 | Probability of any legal score |

The gain is modest here, but the principle is durable: **the objective decides what “best” means**.

Execution consistency matters even more:

| Typical variation | Rough scores out of 100 at the centered aim |
| ---: | ---: |
| 1° | 91 |
| 2° | 61 |
| 3° | 43 |
| 5° | 27 |
| 8° | 17 |

Improving the **mean** moves the centre of the attempt cluster. Improving **consistency** tightens the cluster. Maximum-likelihood training chooses parameters that make observed acceptable outcomes more probable.

#### What just happened

The deterministic target and the likelihood objective preferred different angles:

- 19.13° minimizes distance from one back-net point,
- 20.15° maximizes the chance of landing anywhere inside the legal interval under symmetric noise.

The probability-maximizing aim raises $P(\text{score})$ from about **41.0% to 43.1%** and lowers negative log-likelihood from **0.892 to 0.841**.

That is the direct bridge to machine learning:

> A loss is not merely a scorekeeper. It defines which kind of success the optimizer will pursue.

For classification with a one-hot label, cross-entropy is the negative log probability assigned to the correct class. The Gaussian and softmax distributions are different, but maximum likelihood gives both the same instruction: make the outcomes we observe less surprising.

---

#### Your turn — cost of poor technique

```python
sigma_test = 8.0  # change to 1.0 after predicting the effect
probability_test = probability_of_scoring(probability_aim, sigma_test)
print(f"sigma={sigma_test}°: P(scoring)={probability_test:.1%}")
print(f"negative log-likelihood={-np.log(probability_test + 1e-12):.3f}")
```

This calculation does not claim every football error is Gaussian. It shows how an uncertainty assumption becomes a probability and then a trainable objective.

---

## Summary — One Mental Model for Training

| Idea | Free-kick result | Future ML meaning | Intuition to keep |
| --- | --- | --- | --- |
| Local change | The 25° shot stops rising near 15.6 m | Derivatives measure sensitivity | A derivative is a change meter at one point |
| Scalar descent | 35° → 19.13° | One parameter follows one derivative | Move opposite the direction that increases wrongness |
| Curvature | At 35°, `2/H ≈ 3.56` | Step-size stability depends on how the loss bends | A safe steering move on a gentle bend can be wild on a hairpin |
| Gradient vector | Angle and speed update together | All trainable parameters receive simultaneous instructions | A gradient is one sensitivity per adjustable cause |
| Parameter scaling | Angle uses scale 1.0; speed uses 0.1 | Optimizers rescale coordinates differently | Different units and sensitivities need different effective steps |
| Matrices | Multiple signals mix into multiple outputs | Linear and attention projections | A matrix maps who influences whom |
| Activations | Clip, ReLU, and sigmoid gate signals differently | Non-linear representations and gradient flow | Every gate shapes both forward behavior and backward learning |
| Multi-path chain rule | Wall + target contributions sum at 16° | Backprop adds gradients where graph paths merge | One cause can receive blame through several consequences |
| Likelihood | 19.13° targets a point; 20.15° maximizes P(score) | Loss design and maximum-likelihood training | The objective decides what “best” means |

### The complete loop

1. **Parameters create a forward world.** One angle creates one trajectory; angle and speed together create another.
2. **A loss defines success.** Target precision and scoring likelihood can prefer different solutions.
3. **Derivatives measure local sensitivity.** With many parameters, those measurements form a gradient vector.
4. **Curvature limits useful step size.** The safe update scale depends on how sharply loss bends in the current region.
5. **Backpropagation sends sensitivities through every path.** Contributions add when paths meet at the same parameter.
6. **The optimizer updates all parameters together.** Adaptive optimizers refine the per-parameter scaling idea introduced here.

That cycle — **forward values → chosen objective → backward sensitivities → coordinated update** — is the calculus engine behind model training, control systems, trajectory optimisation, and modern scientific computing.

In [ ]:
# Closing decision: combine optimization, geometry, and uncertainty
final_angle = history[-1]
final_loss = kick_loss(final_angle)
final_wall = ball_height(WALL_X, final_angle)
final_goal = ball_height(GOAL_X, final_angle)
final_net = ball_height(NET_X, final_angle)

print("=" * 68)
print("  CLOSING DECISION — WHAT DID CALCULUS BUY US?")
print("=" * 68)
print("\n  OBJECTIVE 1: hit one chosen back-net point")
print(f"    optimized angle:             {final_angle:.3f}°")
print(f"    wall height:                 {final_wall:.3f}m  (need > {WALL_H + BALL_RADIUS:.2f}m)")
print(f"    goal-plane height:           {final_goal:.3f}m  (legal [{GOAL_BOTTOM:.2f}, {GOAL_TOP:.2f}]m)")
print(f"    back-net impact:             {final_net:.3f}m  (target {TARGET_H:.2f}m)")
print(f"    target error:                {final_net - TARGET_H:+.4f}m")
print(f"    physical loss:               {final_loss:.8f}")
print(f"    P(score) with sigma=3°:      {prob_target_aim:.1%}")

print("\n  OBJECTIVE 2: maximize any legal score under execution noise")
print(f"    probability-maximizing aim:  {probability_aim:.3f}°")
print(f"    legal geometric window:      [{theta_lo:.2f}°, {theta_hi:.2f}°]")
print(f"    P(score) with sigma=3°:      {probability_optimized:.1%}")
print(f"    likelihood improvement:      {probability_optimized - prob_target_aim:+.1%}")

print("\n  FOUNDATION")
print("    one derivative became a gradient vector;")
print("    curvature explained useful step size;")
print("    multiple backward paths added at shared causes;")
print("    and the chosen loss determined which solution counted as best.")

---

## What This Notebook Now Establishes

### Implemented and demonstrated

- Directional alignment through dot products
- Local change and turning points through derivatives
- A physically complete loss with gated constraint penalties
- Scalar gradient descent from a high miss to a target-height shot
- Curvature as the reason learning-rate stability is local
- A two-parameter gradient vector for angle and speed
- Simultaneous updates with different per-parameter step scales
- Matrix transforms as coordinated signal mixing
- Hard clipping, ReLU, and sigmoid as different forward/backward gates
- Explicit reverse-mode chain rule
- Multi-path gradient contributions that add at a shared parameter
- Analytic gradients checked against finite differences
- Point-target optimization compared with likelihood optimization
- Gaussian execution noise and negative log-likelihood

### Deliberately left for later chapters

- Momentum and adaptive optimizer state
- High-dimensional non-convex loss landscapes
- Jacobian and Hessian matrices beyond the one-dimensional curvature intuition
- A general automatic-differentiation engine
- Entropy and KL divergence beyond the likelihood bridge
- Aerodynamic drag, lift, spin, and turbulent knuckleball forces

The intended foundation is now wider than one smooth scalar example, while keeping the same visible causal loop.

---

## When to Reach for Each Idea

| Question | Tool | What it answers |
| --- | --- | --- |
| “How aligned are these directions?” | Dot product | How strongly one vector lies along another |
| “How fast is this changing here?” | Derivative | Local output change per unit input change |
| “Which way should one parameter move?” | Scalar gradient descent | Opposite the local slope of loss |
| “How should many parameters move together?” | Gradient vector | One simultaneous instruction per parameter |
| “Why did this learning rate bounce?” | Curvature | The loss bent too sharply for that step |
| “How do several inputs influence several outputs?” | Matrix multiply | All weighted signal routes at once |
| “Which signals pass, stop, or saturate?” | Activation function | Forward gating and backward gradient behavior |
| “How did this final error come from earlier values?” | Chain rule | Local sensitivities multiplied backward |
| “What if several paths reach the same parameter?” | Multi-path backprop | Add every path's gradient contribution |
| “What kind of success will training pursue?” | Loss design | The objective's definition of best |
| “How reliable is a solution under noise?” | Probability and likelihood | Mass assigned to acceptable outcomes |

---

## What's Next

The next chapters reuse these ideas with learned weights instead of kick controls:

| Foundation from here | Reappears as |
| --- | --- |
| Scalar derivative | Sensitivity of loss to one weight |
| Gradient vector | Simultaneous update for every model parameter |
| Per-knob step scales | Adam's adaptive effective learning rates |
| Curvature intuition | Why training can oscillate, crawl, or need schedules |
| Matrix mixing | Dense layers and transformer projections |
| ReLU / sigmoid gates | Hidden activations and probability-producing heads |
| Multi-path addition | Residual connections and branching computation graphs |
| Likelihood objective | Regression likelihoods and classification cross-entropy |

The durable mental model is:

> **Run the world forward. Define what success means. Measure how each cause affects wrongness. Send those sensitivities backward through every path. Update all causes together at useful scales.**

→ **Next:** [`../01-ml-basics/ml-basics.ipynb`](../01-ml-basics/ml-basics.ipynb) — linear regression and classification from scratch, where the single kick controls become learned parameter vectors.